# 01 · Build & Clean the Province-Month Panel

Loads ACLED + GTD, harmonises province names across the two independent
sources, aggregates each to a province-month panel, merges onto the
18 × 60 skeleton (Turkish strikes as an exogenous control), adds logs and
distributed lags, and saves `data/panel_iraq_2016_2020.csv`.

> Requires `data/ACLED.csv` and `data/GTD.xlsx`.

In [6]:
# --- Setup: run from repo root so `data/` and `src/` paths resolve ---
import sys, os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")            # step up from notebooks/ to the repo root
sys.path.insert(0, "src")     # make the modules importable

%load_ext autoreload
%autoreload 2

print("Working directory:", os.getcwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working directory: /Users/fizarizvi/Desktop/iraq_repo


## Load & clean the two sources

In [7]:
import build_panel as bp

acled, coalition_isis = bp.load_acled()   # data/ACLED.csv -> Coalition-vs-ISIL strikes
gtd_iq = bp.load_gtd()                     # data/GTD.xlsx  -> Iraq attacks, cleaned

Coalition-vs-ISIL strikes: 3924
GTD Iraq 2016-2021 attacks: 8689


## Aggregate to province-month panels

In [8]:
gtd_pm     = bp.build_gtd_pm(gtd_iq)      # outcome side
acled_pm   = bp.build_acled_pm(coalition_isis)  # treatment side
turkish_pm = bp.build_turkish_pm(acled)  # exogenous control

GTD province-months: 617
ACLED province-months: 257
Turkish strike province-months: 175


## Assemble the merged panel + save

In [9]:
panel = bp.assemble_panel(gtd_iq, gtd_pm, acled_pm, turkish_pm)
panel.to_csv("data/panel_iraq_2016_2020.csv", index=False)
panel.head()

Panel size: 18 x 60 = 1080
Final panel: 1080 rows | usable after 6 lags: 972


,province,year_month,n_attacks,n_isis_attacks,n_isis_success,n_killed,n_wounded,n_casualties,n_attacks_with_casualties,n_suicide,...,province_id,time_index,log_strikes,log_turkish_strikes,log_strikes_lag1,log_strikes_lag2,log_strikes_lag3,log_strikes_lag4,log_strikes_lag5,log_strikes_lag6
0,Al Anbar,2016-01-01,40,22,15.0,496,207,570,33,17,...,0,0,4.143135,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Al Anbar,2016-02-01,91,43,27.0,485,125,477,82,30,...,0,1,4.595120,0.0,4.143135,NaN,NaN,NaN,NaN,NaN
2,Al Anbar,2016-03-01,70,32,22.0,301,137,386,66,20,...,0,2,4.605170,0.0,4.595120,4.143135,NaN,NaN,NaN,NaN
3,Al Anbar,2016-04-01,56,25,17.0,200,199,377,47,15,...,0,3,4.770685,0.0,4.605170,4.595120,4.143135,NaN,NaN,NaN
4,Al Anbar,2016-05-01,111,37,26.0,352,151,369,102,27,...,0,4,5.117994,0.0,4.770685,4.605170,4.595120,4.143135,NaN,NaN


## Sanity checks
Confirm the panel matches the dissertation's input figures before moving on.

In [10]:
assert panel.shape[0] == 1080, f"expected 1080 rows, got {panel.shape[0]}"
assert panel['province'].nunique() == 18, "expected 18 provinces"
assert 'n_isis_success' in panel.columns, "n_isis_success missing -- update build_panel.py"

print("Provinces:", panel['province'].nunique())
print("Rows:", len(panel), "(18 x 60 = 1080)")
print("GTD total attacks:", panel['n_attacks'].sum(), "(expect 8689)")
print("ACLED total strikes:", panel['n_strikes'].sum(), "(expect 3924)")
print("n_isis_success column present:", 'n_isis_success' in panel.columns)

Provinces: 18
Rows: 1080 (18 x 60 = 1080)
GTD total attacks: 8689 (expect 8689)
ACLED total strikes: 3881 (expect 3924)
n_isis_success column present: True
